# Model server — use Qwen from the app on your PC (Kaggle notebook)

Loads Qwen2.5-7B on Kaggle's GPUs and opens a temporary private link. The app on your PC (Streamlit +
API) sends the CV text to it and gets the extracted profile back. Reading the document and the OCR stay
on your PC.

**Once:** import this notebook on kaggle.com (*Create → New Notebook → File → Import Notebook*), then in
the right panel: **Accelerator: GPU T4 x2**, **Internet: On**, and *Add Input → Your Work → Datasets →
`recruitment-ai-eval-v1`* (our code; refresh it with `python scripts/kaggle_run.py upload`).

**Each time:**
1. *Run All*. After about 4 minutes the last cell prints two lines, `REMOTE_LLM_URL=...` and
   `REMOTE_LLM_TOKEN=...`.
2. Paste them into `.env` on your PC, with `LLM_PROVIDER=remote`, and (re)start the API
   (`python -m uvicorn api.main:app`) and the app (`python -m streamlit run frontend/streamlit_app.py`).
3. In the app: *Candidates* → upload a CV → *Extract profile*.
4. **When you are done, stop the session** (power icon, top right). The GPU quota counts while the
   session is open, even idle. The server also stops by itself after `MAX_HOURS`, or `IDLE_MINUTES`
   without a request, but only stopping the session ends the GPU time.

The link and the token change at every session. Only use CVs you are allowed to process (your own, or
invented ones): their text goes to this Kaggle session.

In [ ]:
# ════════════ 1. PARAMETERS ════════════
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # or "Qwen/Qwen2.5-VL-7B-Instruct" (reads page images)
ADAPTER = ""          # "" = base model; "auto" = the fine-tuned adapter of an attached finetune output
MAX_HOURS = 2.0       # the server stops by itself after this long...
IDLE_MINUTES = 30     # ...or after this long without any request (protects the GPU quota)
PORT = 8000

In [ ]:
# ════════════ 2. Setup: GPU check, libraries, project code ════════════
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

import torch

gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", gpus or "none")
if not gpus or any("P100" in g for g in gpus):
    raise SystemExit("Settings → Accelerator → 'GPU T4 x2', then run again.")
subprocess.run("pip install -q -U 'transformers>=4.56' 'accelerate>=1.0' 'peft>=0.13' fastapi uvicorn "
               "pydantic-settings email-validator", shell=True, check=True)

PROJECT = Path("/tmp/project")
if not (PROJECT / "src").is_dir():
    archive = next(Path("/kaggle/input").rglob("project.tar.gz"), None)
    if archive:
        with tarfile.open(archive) as tar:
            tar.extractall(PROJECT.parent, filter="data")
    else:
        found = next((p for p in Path("/kaggle/input").rglob("project") if (p / "src").is_dir()), None)
        if found is None:
            raise SystemExit("Project code not found: Add Input → Your Work → Datasets → recruitment-ai-eval-v1")
        shutil.copytree(found, PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
print("✅ ready: project code in", PROJECT)

In [ ]:
# ════════════ 3. Load the model on the GPUs (~3 min) ════════════
os.environ.update({"LLM_DEVICE": "cuda", "LLM_DTYPE": "float16",  # T4: no bfloat16
                   "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
from recruitment_ai.llm.huggingface_adapter import HuggingFaceAdapter

adapter = None
if ADAPTER == "auto":
    adapter = next((str(p.parent) for p in Path("/kaggle/input").rglob("adapter/adapter_config.json")), None)
    if adapter is None:
        raise SystemExit("ADAPTER = 'auto' but no fine-tuned adapter is attached (Add Input → the finetune output)")
elif ADAPTER:
    adapter = ADAPTER
llm = HuggingFaceAdapter(model=MODEL_ID, adapter=adapter)
llm._ensure_loaded()
print("test answer:", llm.generate("Answer with one word: OK", max_tokens=5).content)
print(f"✅ model loaded: {MODEL_ID}" + (f" + adapter {adapter}" if adapter else ""))

In [ ]:
# ════════════ 4. Start the server and the private link ════════════
import re, secrets, threading
import httpx
import uvicorn
from recruitment_ai.llm.server import create_app

TOKEN = secrets.token_urlsafe(24)
server = uvicorn.Server(uvicorn.Config(create_app(llm, TOKEN), host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(60):
    try:
        httpx.get(f"http://127.0.0.1:{PORT}/info", headers={"Authorization": f"Bearer {TOKEN}"}, timeout=2)
        break
    except httpx.HTTPError:
        time.sleep(1)

# Cloudflare "quick tunnel": a temporary https link to this session, no account needed
if not Path("/tmp/cloudflared").exists():
    subprocess.run("curl -sL -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/"
                   "download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:  # the link appears in cloudflared's first lines
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        URL = match.group(0)
        break
if URL is None:
    raise SystemExit("The tunnel did not start: run this cell again")
# keep reading cloudflared's output in the background, otherwise it would block when its buffer is full
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
time.sleep(5)  # the link takes a few seconds to work everywhere
print("\nPaste these lines into .env on your PC, then (re)start the API and the app:\n")
print("LLM_PROVIDER=remote")
print(f"REMOTE_LLM_URL={URL}")
print(f"REMOTE_LLM_TOKEN={TOKEN}")

In [ ]:
# ════════════ 5. Keep serving (stops by itself; then STOP THE SESSION) ════════════
start = time.time()
last_count, last_activity = 0, time.time()
try:
    while True:
        time.sleep(60)
        info = httpx.get(f"http://127.0.0.1:{PORT}/info", headers={"Authorization": f"Bearer {TOKEN}"}).json()
        if info["requests"] != last_count:
            last_count, last_activity = info["requests"], time.time()
        hours, idle = (time.time() - start) / 3600, (time.time() - last_activity) / 60
        print(f"{time.strftime('%H:%M')} | {info['requests']} requests | idle {idle:.0f} min", flush=True)
        if hours >= MAX_HOURS or idle >= IDLE_MINUTES:
            break
except KeyboardInterrupt:  # the stop button of the cell
    pass
tunnel.terminate()
server.should_exit = True
print("Server stopped. Now STOP THE SESSION (power icon, top right): the GPU quota counts until you do.")